In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.stats import gaussian_kde

mpl.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['font.size'] = 15

In [ ]:
log_p = df["log_probability"]
counts = df["count"]
values = np.vstack([log_p,counts])

In [ ]:
kde = gaussian_kde(values)

# Grid für Dichte
xmin, xmax = -1200, 0
ymin, ymax = 0, 105
xx, yy = np.mgrid[xmin:xmax:200j, ymin:ymax:200j]
grid_coords = np.vstack([xx.ravel(), yy.ravel()])
zz = kde(grid_coords).reshape(xx.shape)

# Dichte-Quantile (75%, 50%, 25%)
z_sorted = np.sort(zz.ravel())[::-1]
z_cumsum = np.cumsum(z_sorted)
z_cumsum /= z_cumsum[-1]

levels = [
    z_sorted[np.searchsorted(z_cumsum, 0.75)],
    z_sorted[np.searchsorted(z_cumsum, 0.50)],
    z_sorted[np.searchsorted(z_cumsum, 0.25)],
]

In [ ]:
fig1, ax1 = plt.subplots(1,2, figsize=(12,5), dpi=100)
ax1[0].hist(log_p, bins=15, color="black", density=True, histtype="step")
ax1[0].set_xlim(xmin, xmax)
ax1[0].set_xlabel(r"log(p)")
ax1[0].set_ylabel("Normalized distribution")
ax1[0].tick_params(labelleft=False)
ax1[0].grid(linestyle="--")
ax1[0].set_axisbelow(True)

ax1[1].scatter(log_p, counts, color="black", label="QCD")
ax1[1].plot(log_p, -log_p/np.log(41*31*31), color="grey", linestyle="--")
ax1[1].set_xlim(xmin, xmax)
ax1[1].set_ylim(ymin, ymax)
ax1[1].set_xlabel(r"log(p)")
ax1[1].set_ylabel("Multiplicity")
ax1[1].grid(linestyle="--")
ax1[1].set_axisbelow(True)
ax1[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(7,7), dpi=100)
gs = GridSpec(
    2, 2,
    width_ratios=[2.5, 1],
    height_ratios=[1, 2.5],
    hspace=0.04,
    wspace=0.04
)

bins=15

# Achsen definieren
ax_histx = fig.add_subplot(gs[0, 0])
ax_main = fig.add_subplot(gs[1, 0])
ax_histy = fig.add_subplot(gs[1, 1])


contours = ax_main.contour(xx, yy, zz, levels=levels, colors=["lightgrey", "grey", "black"])
#ax_main.scatter(log_p, counts, color="black", marker=".")
ax_main.set_xlim(xmin, xmax)
ax_main.set_ylim(ymin, ymax)
ax_main.set_xlabel(r"log(p)")
ax_main.set_ylabel("Multiplicity")
ax_main.grid(linestyle="--")
ax_main.set_axisbelow(True)
ax_main.scatter(1,0, color="black", label="QCD", marker=".")
ax_main.legend(loc="upper right")

# Histogramm oben (x-Achse)
ax_histx.hist(log_p, bins=bins, color="black", histtype="step", density=True)
ax_histx.set_xlim(ax_main.get_xlim())
ax_histx.set_ylabel("Normalized\ndistribution")
ax_histx.tick_params(labelbottom=False)
ax_histx.tick_params(labelleft=False)
ax_histx.grid(linestyle="--")
ax_histx.set_axisbelow(True)

# Histogramm rechts (y-Achse)
ax_histy.hist(counts, bins=bins, orientation="horizontal", color="black", histtype="step", density=True)
ax_histy.set_ylim(ax_main.get_ylim())
ax_histy.set_xlabel("Normalized\ndistribution")
ax_histy.tick_params(labelleft=False)
ax_histy.tick_params(labelbottom=False)
ax_histy.grid(linestyle="--")
ax_histy.set_axisbelow(True)

#plt.tight_layout()
plt.show()